# <center>Big Data – Exercises</center>
## <center>Fall 2020 – Week 13 – ETH Zurich</center>

## Introduction

In this exercise, we analyze the sales data of a fictious wholesale supplier
(taken from the database system benchmark [TPC-H](http://www.tpc.org/tpch/))
in our favorite spreadsheet application. Then, we will use SQL to query the data shape cube. 
A cube is a collection of numeric data organized by arrays of discrete identifiers (Janus and Fouché, 2009). It is quite natural, as we see in the lectures, to map cubes to tables. 

## 1. The TPC-H Dataset as OLAP Cube

Let us get familiar with the dataset.
It consists of orders made, each of which is made by a customer, and consists of lineitems.
Think of an order as a shopping cart with several items in it.
The items of an order are parts that may be provided by different suppliers.
Suppliers and customers come from different nations, which are grouped into regions of the world.
The following figure illustrates the schema of the TPC-H dataset.
<br>

![Schema of the TPC-H dataset](https://cloud.inf.ethz.ch/s/MNY8ksxgX78zf9a/download)


### Task 1

1. Which table(s) of the TPC-H schema is/are the fact table(s)?
1. What is/are the measure(s)?
1. What are the dimensions?
1. What do you call this flavor of OLAP?

## 2. Analyzing TPC-H with a Pivot Table

Download [this Excel file](https://cloud.inf.ethz.ch/s/rJKGqkHxbAtYQog/download)
and open it with your favorite spreadsheet application.
The file contains a universal table (a fully denormalized table) of a small TPC-H dataset.
The schema has been modified slightly to make analysis in a spreadsheet application easier:
The two precomputed measures revenue and cost
as well as the hierarchy of time dimensions in the attribute *orderdate* have been added in the materialized form
and some other attributes have been removed.

You may need to look up how to use pivot tables in your spreadsheet application.

1. Microsoft Excel: [PivotTable](https://support.office.com/en-us/article/Create-a-PivotTable-to-analyze-worksheet-data-a9a84538-bfe9-40a9-a8e9-f99134456576)
1. Google Sheets: [pivot tables](https://support.google.com/docs/answer/1272900?co=GENIE.Platform%3DDesktop&hl=en)
1. Open Office: [DataPilot](https://openoffice.blogs.com/openoffice/2006/11/data_pilots_in_.html)

### Task 1: Discussion

Discuss the terms "slice and dice", "drill down", "roll up", and "pivoting".

### Task 2: Create the following pivot tables:

1. Show how much revenue suppliers from different regions (as columns) produced in every year (as rows).
1. Show how much revenue suppliers from nations of Africa (as columns) produced in every year (as rows).
1. Show how much revenue suppliers from nations of Africa produced in every quarter of every year.
1. Show how much revenue suppliers from nations of Africa produced in every week of every month of Q1 in 1996.
1. Show how much revenue suppliers from nations of Africa produced in every year with "urgent" orders.
1. Show the average order quantity for parts from suppliers from nations of Africa per year.
1. Show how much revenue suppliers from nations of Africa (as rows) produced in every quarter of every year (as columns).

## 3. OLAP Cubes and SQL

### Part 1: SQL

Write SQL queries for the PivotTables from Question 2 Task 1.

#### Notes

* Assume that the revenue is calculated as `olquantity * partretailprice * (1-oldiscount)`.
* To get the year or quarter from a date in PostgreSQL, you can use [`DATE_PART ("field", date )  `](https://www.postgresqltutorial.com/postgresql-date_part/). Note that the field is case-insensitive. You can write `DATE_PART("YEAR", date)` or `DATE_PART("year", date)`, which are equivalent. 

#### Connect to our SQL server

Execute the following cells in order to be able to run SQL cells later.

In [ ]:
!pip install psycopg2
!pip install ipython-sql

Now we connect to the DB we set as publically available for this exercise with the following credentials.

In [ ]:
server='ethbigdata2020.postgres.database.azure.com'
user='student@ethbigdata2020'
password='BigData2020'
database='tpch-db'

In [ ]:
%load_ext sql
connection_string = f'postgresql://{user}:{password}@{server}:5432/{database}?sslmode=require'
%sql $connection_string

First however, we define the fact table using a WITH statement (copy this at the beginning of all other queries)

In [ ]:
%%sql
WITH cube AS (
    SELECT olquantity, partretailprice, oldiscount,
           orderdate, nationname, regionname, orderpriority,
           olquantity * partretailprice * (1-oldiscount) AS revenue,
           DATE_PART('year', orderdate) AS orderyear,
           DATE_PART('quarter', orderdate) AS orderquarter,
           DATE_PART('month', orderdate) AS ordermonth,
           DATE_PART('week', orderdate) AS orderweek
    FROM orderline ol
    JOIN orders o      ON ol.orderid = o.orderid
    JOIN supplypart sp ON ol.partId = sp.partId AND ol.supplierId = sp.supplierId
    JOIN part p        ON sp.partId = p.partId
    JOIN supplier s    ON sp.supplierId = s.supplierId
    JOIN nation sn     ON s.nationId = sn.nationId
    JOIN region sr     ON sn.regionId = sr.regionId
)
SELECT * FROM cube LIMIT 10

Note that, for the purpose of this exercise, we dropped some dimensions of the cube because none of the queries uses them. Also, we materialize some hierarchy levels of the `orderdate` dimension in order to make the subsequent queries more readable. This makes them *look* like they were new dimensions -- conceptually, they are not! (They are, well, levels of a hierarchy of the `orderdate` dimension.)

OK, you are good to go. Use the SQL cell below and add more as you need.

Note that the numbers you obtain with the SQL queries should not be identical to those in the pivot tables in Task2, because the data we have in the DB have more rows in its fact table.

#### Your Answers

In [ ]:
%%sql
WITH cube AS (
    SELECT olquantity, partretailprice, oldiscount,
           orderdate, nationname, regionname, orderpriority,
           olquantity * partretailprice * (1-oldiscount) AS revenue,
           DATE_PART('year', orderdate) AS orderyear,
           DATE_PART('quarter', orderdate) AS orderquarter,
           DATE_PART('month', orderdate) AS ordermonth,
           DATE_PART('week', orderdate) AS orderweek
    FROM orderline ol
    JOIN orders o      ON ol.orderid = o.orderid
    JOIN supplypart sp ON ol.partId = sp.partId AND ol.supplierId = sp.supplierId
    JOIN part p        ON sp.partId = p.partId
    JOIN supplier s    ON sp.supplierId = s.supplierId
    JOIN nation sn     ON s.nationId = sn.nationId
    JOIN region sr     ON sn.regionId = sr.regionId
)

SELECT * FROM cube

### Part 2: MDX (Optional)

Choose one of the queries you wrote in SQL and implement it in MDX.